# ✂️ Text Splitters and Chunking Strategies

## Learning Objectives
In this notebook, you will learn:
1. **Recursive Character Splitting** - the default, general-purpose way to break text into chunks using a cascade of separators
2. **Chunk Size & Overlap Tradeoffs** - how `chunk_size` and `chunk_overlap` affect the number, size, and continuity of chunks
3. **Structure-Aware Splitting** - splitting Markdown by header hierarchy and source code by language-aware boundaries
4. **Document-Level Splitting** - applying a splitter to loaded `Document` objects (e.g., from a PDF) while preserving metadata

## Prerequisites
- Basic understanding of RAG pipelines and why chunking matters for retrieval quality
- Familiarity with LangChain's `Document` object
- `langchain-text-splitters` and `pypdf` installed (`pip install langchain-text-splitters pypdf`)
- A `.env` file is not strictly required here (no LLM calls are made), but `load_dotenv()` is kept for consistency with the rest of this course


---
## 🔧 Part 0: Environment Setup

We load environment variables (for consistency with the rest of the course, even though this
notebook makes no LLM calls) and import the text-splitter classes we'll compare throughout
the notebook.

In [ ]:
# ============================================================================
# ENVIRONMENT SETUP: Imports
# ============================================================================
from dotenv import load_dotenv

from langchain_core.documents import Document
from langchain_text_splitters import (
    CharacterTextSplitter,
    Language,
    MarkdownHeaderTextSplitter,
    RecursiveCharacterTextSplitter,
    TokenTextSplitter,
)

load_dotenv()

print("✅ Environment loaded and text-splitter classes imported!")


### 📄 Sample Documents

We use two small fixtures throughout the notebook: a Markdown-formatted article
(`SAMPLE_TEXT`) to exercise the character- and structure-aware splitters, and a short
Python source file (`SAMPLE_CODE`) to exercise the language-aware code splitter.

In [ ]:
# ============================================================================
# SAMPLE DATA: Markdown Article and Python Source Fixtures
# ============================================================================
SAMPLE_TEXT = """# Introduction to Machine Learning

Machine learning is a subset of artificial intelligence that enables systems to learn and improve from experience without being explicitly programmed.

## Types of Machine Learning

### Supervised Learning
Supervised learning uses labeled data to train models. The algorithm learns to map inputs to outputs based on example input-output pairs.

Common algorithms include:
- Linear Regression
- Decision Trees
- Neural Networks

### Unsupervised Learning
Unsupervised learning finds hidden patterns in unlabeled data. The algorithm discovers structure without predefined labels.

Common algorithms include:
- K-Means Clustering
- Principal Component Analysis
- Autoencoders

## Applications

Machine learning is used in many fields:
1. Image recognition
2. Natural language processing
3. Recommendation systems
4. Fraud detection
5. Autonomous vehicles
""".strip()

SAMPLE_CODE = '''
def quicksort(arr):
    """
    Quicksort implementation in Python.
    Time complexity: O(n log n) average, O(n²) worst case.
    """
    if len(arr) <= 1:
        return arr

    pivot = arr[len(arr) // 2]
    left = [x for x in arr if x < pivot]
    middle = [x for x in arr if x == pivot]
    right = [x for x in arr if x > pivot]

    return quicksort(left) + middle + quicksort(right)


def binary_search(arr, target):
    """
    Binary search implementation.
    Requires sorted array.
    Time complexity: O(log n)
    """
    left, right = 0, len(arr) - 1

    while left <= right:
        mid = (left + right) // 2
        if arr[mid] == target:
            return mid
        elif arr[mid] < target:
            left = mid + 1
        else:
            right = mid - 1

    return -1
'''

print(f"📄 SAMPLE_TEXT: {len(SAMPLE_TEXT)} chars")
print(f"📄 SAMPLE_CODE: {len(SAMPLE_CODE)} chars")


---
## ✂️ Part 1: Character-Based Splitting

`RecursiveCharacterTextSplitter` is the default, general-purpose splitter in LangChain. It
tries a cascade of separators (paragraph, then line, then space, then character) and only
falls back to a coarser separator when a chunk is still too big — this keeps semantically
related text (like a paragraph) together whenever possible.

### Key Concepts:
- **`chunk_size`**: The target maximum size (in characters, by default) of each chunk
- **`chunk_overlap`**: How many characters of a chunk are repeated at the start of the next chunk, to preserve context across a chunk boundary
- **`separators`**: An ordered list of strings to split on, tried from most- to least- semantically meaningful

### 1.1 🔁 Recursive Character Splitter

Split `SAMPLE_TEXT` with a fixed `chunk_size`/`chunk_overlap` and inspect the resulting chunk sizes.

In [ ]:
# ============================================================================
# RECURSIVE SPLITTER: Split a Markdown Document by Character Count
# ============================================================================
def recursive_splitter():
    splitter = RecursiveCharacterTextSplitter(
        chunk_size=500,
        chunk_overlap=50,
        separators=["\n\n", "\n", " ", ""],  # Tried in order: paragraph -> line -> word -> char
    )
    chunks = splitter.split_text(SAMPLE_TEXT)

    print(f"Original length: {len(SAMPLE_TEXT)} chars")
    print(f"Number of chunks: {len(chunks)}")
    print(f"Chunk sizes: {[len(c) for c in chunks]}")
    print(f"\nFirst chunk preview:\n{chunks[0][:200]}...")


### 1.2 📏 Chunk Size Comparison

Sweeping `chunk_size` shows the direct tradeoff between chunk granularity and chunk count — smaller chunks mean more, more-focused chunks; larger chunks mean fewer, more context-rich chunks.

In [ ]:
# ============================================================================
# CHUNK SIZE COMPARISON: Effect of chunk_size on Chunk Count
# ============================================================================
def chunk_size_comparison():
    sizes = [200, 500, 1000]

    print("=== Chunk Size Comparison ===")
    for size in sizes:
        splitter = RecursiveCharacterTextSplitter(
            chunk_size=size, chunk_overlap=size // 5
        )  # 20% overlap, scaled with chunk size
        chunks = splitter.split_text(SAMPLE_TEXT)
        print(f" Size {size}: {len(chunks)} chunks")


### 1.3 🔗 Why Overlap Matters

Without overlap, a sentence or phrase that straddles a chunk boundary gets cut in half, which can hurt retrieval and generation quality. This demo compares chunk boundaries with and without `chunk_overlap`.

In [ ]:
# ============================================================================
# OVERLAP IMPORTANCE: Comparing Chunk Boundaries With and Without Overlap
# ============================================================================
def overlap_importance():
    text = "The quick brown fox jumps over the lazy dog. " * 10  # Repeated text

    # Without overlap: chunk boundaries are hard cuts
    no_overlap = RecursiveCharacterTextSplitter(chunk_size=50, chunk_overlap=0)

    # With overlap: the tail of one chunk reappears at the head of the next
    with_overlap = RecursiveCharacterTextSplitter(chunk_size=50, chunk_overlap=20)

    chunks_no_overlap = no_overlap.split_text(text)
    chunks_with_overlap = with_overlap.split_text(text)

    print("Without overlap:")
    print(f"  Chunk 1 end: ...{chunks_no_overlap[0][-20:]}")
    print(f"  Chunk 2 start: {chunks_no_overlap[1][:20]}...")

    print("\nWith overlap:")
    print(f"  Chunk 1 end: ...{chunks_with_overlap[0][-20:]}")
    print(f"  Chunk 2 start: {chunks_with_overlap[1][:20]}...")


---
## 🧩 Part 2: Structure-Aware Splitting

Plain character counting ignores the structure of the document. When the source has known
structure — Markdown headers, or a programming language's syntax — a structure-aware splitter
produces chunks that respect that structure, which usually yields more coherent, more
retrievable chunks than blind character splitting.

### Key Insight:
> Structure-aware splitters trade a purely size-driven cut for a boundary that respects the
> document's own semantics (a heading section, a function definition). Prefer them whenever
> your source format has exploitable structure.

### 2.1 📑 Markdown Header Splitter

`MarkdownHeaderTextSplitter` splits on Markdown header levels (`#`, `##`, `###`, ...) and attaches the header path each chunk fell under as chunk metadata — useful for keeping section context alongside the chunk text.

In [ ]:
# ============================================================================
# MARKDOWN SPLITTER: Split by Header Hierarchy (H1 -> H2 -> H3)
# ============================================================================
def markdown_splitter():
    headers_to_consider = [
        ("#", "h1"),
        ("##", "h2"),
        ("###", "h3"),
    ]
    splitter = MarkdownHeaderTextSplitter(headers_to_split_on=headers_to_consider)
    chunks = splitter.split_text(SAMPLE_TEXT)

    print(f"Markdown Splitter produced {len(chunks)} chunks.")
    for i, chunk in enumerate(chunks):
        print(f"--- Chunk {i} ---")
        print(f" Metadata: {chunk.metadata}\n")
        print(f" Content: {chunk.page_content[:200]}...\n")


### 2.2 💻 Code-Aware Splitter

`RecursiveCharacterTextSplitter.from_language()` supplies language-specific separators (e.g. `class`/`def` boundaries for Python) so chunks tend to break between functions rather than mid-function.

In [ ]:
# ============================================================================
# CODE SPLITTER: Language-Aware Splitting of Python Source
# ============================================================================
def code_splitter():
    python_splitter = RecursiveCharacterTextSplitter.from_language(
        language=Language.PYTHON, chunk_size=500, chunk_overlap=50
    )
    chunks = python_splitter.split_text(SAMPLE_CODE)
    print(f"Code Splitter produced {len(chunks)} chunks.")
    for i, chunk in enumerate(chunks):
        print(f"\nChunk {i} ({len(chunk)} chars):")
        print(chunk[:150] + "..." if len(chunk) > 150 else chunk)


---
## 📚 Part 3: Splitting Loaded Documents

In a real RAG pipeline you rarely split raw strings — you split `Document` objects produced
by a loader (PDF, HTML, etc.), and the splitter must **preserve and propagate the source
metadata** (e.g. page numbers) onto every resulting chunk.

> **Note**: This demo loads `./docs/langchain_demo.pdf` with `PyPDFLoader`, which requires
> the `pypdf` package and a PDF at that relative path. Adjust the path if you run this
> notebook from a different working directory.

In [ ]:
# ============================================================================
# DOCUMENT SPLITTER: Split Loaded PDF Documents While Preserving Metadata
# ============================================================================
def document_splitter():
    from langchain_community.document_loaders import PyPDFLoader

    loader = PyPDFLoader("./docs/langchain_demo.pdf")
    docs = loader.load()

    print(f"Loaded {len(docs)} documents from PDF.")

    splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=50)

    # split_documents() (as opposed to split_text()) carries each source Document's
    # metadata (e.g. page number) forward onto every chunk it produces
    split_docs = splitter.split_documents(docs)

    print(f"Split into {len(split_docs)} chunks")
    print(f"\nFirst chunk metadata: {split_docs[0].metadata}")
    print(f"First chunk content: {split_docs[0].page_content[:200]}...")
    print(f"\nLast chunk metadata: {split_docs[-1].metadata}")


---
## ▶️ Run

The original `__main__` guard is kept verbatim below — Jupyter sets `__name__` to
`"__main__"`, so this cell runs as-is. Uncomment a line to run that demo (only one demo is
enabled by default: `document_splitter()`, which needs `pypdf` and a PDF at
`./docs/langchain_demo.pdf`).

In [ ]:
# ============================================================================
# RUN: Execute a Chosen Demo (mirrors the original script's `if __name__` guard)
# ============================================================================
if __name__ == "__main__":
    # print("=== Recursive Character Text Splitter ===")
    # recursive_splitter()
    # chunk_size_comparison()
    # overlap_importance()
    # print("=== Markdown Header Text Splitter ===")
    # markdown_splitter()
    # print("=== Code Splitter ===")
    # code_splitter()
    print("=== Document Splitter from PDF ===")
    document_splitter()


---
## 📝 Summary

In this notebook, we compared several chunking strategies for RAG and how each one trades
off chunk coherence, count, and metadata preservation.

### 1. Character-Based Splitting
- **`RecursiveCharacterTextSplitter`**: The default, general-purpose splitter — cascades through separators (paragraph -> line -> word -> char) to keep related text together
- **`chunk_size`**: Larger values mean fewer, more context-rich chunks; smaller values mean more, more-focused chunks
- **`chunk_overlap`**: Repeats a slice of each chunk at the start of the next to avoid hard cuts through mid-sentence context

### 2. Structure-Aware Splitting
- **`MarkdownHeaderTextSplitter`**: Splits on header hierarchy and attaches the header path as chunk metadata
- **`RecursiveCharacterTextSplitter.from_language()`**: Uses language-specific separators (e.g. `def`/`class` boundaries) so code chunks avoid breaking mid-function

### 3. Document-Level Splitting
- **`split_documents()`** vs **`split_text()`**: Use `split_documents()` on loaded `Document` objects so each chunk keeps the source metadata (e.g. page number) from the document it came from

### Next Steps
- Explore the query-transformation and hybrid-search notebooks in this phase to see how chunk quality affects retrieval
- Try `TokenTextSplitter` (imported but unused above) when you need chunk boundaries measured in LLM tokens rather than characters
